- architecture (5 conv+pool blocks + final conv)
- loss function (weighted MSE for confidence and localization)
- mAP metric calculation

# Grid-Based Detection (like YOLO)

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import plotly.graph_objects as go
import pandas as pd


anchors = [0.01]
```
This is a **reference width**. Think of it as: "peaks are typically about 1% of the signal width."

The model predicts a **scaling factor** to adjust this: 
- scaling_factor = 2.0 means the peak is twice the anchor width
- scaling_factor = 0.5 means half the anchor width

### 2. The Model Architecture
```
Input (1024 points) 
    ↓
[Conv + Pool] → reduces to 512 points
[Conv + Pool] → reduces to 256 points
[Conv + Pool] → reduces to 128 points
[Conv + Pool] → reduces to 64 points
[Conv + Pool] → reduces to 32 points (32 cells!)
[Conv] → 3 outputs per cell
    ↓
Output (32 cells × 3 values)

In [5]:
# Magic numbers
max_signal_length = 1024
max_signal_height = 255
max_n_peaks = 3
n_info_values = 11

# Hyperparameters
anchors = [0.01]
n_anchors = len(anchors)
n_cells = 32
n_outputs = 3

In [6]:
# Data loading functions
def load_raw_covision_data(info_path: str, signal_path: str):
    """Load raw data from info and signal files."""
    info_data = np.fromfile(info_path, dtype=np.float32)
    signal_data = np.fromfile(signal_path, dtype=np.ubyte)
    return info_data, signal_data


def raw_covision_data_to_examples(info_data: np.array, signal_data: np.array):
    """Convert raw data to a more convenient form."""
    examples = []
    
    info_data = info_data.reshape((-1, n_info_values))
    signal_data = signal_data.reshape((-1, max_signal_length))
    
    for example_info_data, example_signal_data in zip(info_data, signal_data):
        signal_length = int(example_info_data[0])
        n_peaks = int(example_info_data[1])
        
        signal = example_signal_data[:signal_length].astype(np.float32)
        peaks = example_info_data[2:].reshape((max_n_peaks, -1))[:n_peaks]
        
        examples.append({"signal": signal, "peaks": peaks})
    
    return examples

In [ ]:
def display_example(example):
    """Display signal with peaks using Plotly."""
    plots = [
        go.Scatter(
            x=np.arange(len(example["signal"])),
            y=example["signal"],
            mode="lines",
            showlegend=False,
        )
    ]
    
    for peak in example["peaks"]:
        position = peak[0]
        height = peak[1]
        width = peak[2]
        start = position - width / 2
        stop = start + width
        plots.append(
            go.Scatter(
                x=[start, stop],
                y=[height, height],
                fill="tozeroy",
                mode="none",
                fillcolor="rgba(255, 0, 0, 0.5)",
                showlegend=False,
            ),
        )
    
    fig = go.Figure(data=plots)
    fig.show()


In [ ]:

# PyTorch Dataset
class CovisionDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        example = self.examples[idx]
        
        # Prepare input
        x = np.zeros((max_signal_length, 1), dtype=np.float32)
        signal_length = len(example["signal"])
        x[:signal_length, 0] = example["signal"]
        
        # Prepare target
        y = np.zeros((n_cells, n_anchors * n_outputs), dtype=np.float32)
        
        for (pos, height, width) in example["peaks"]:
            norm_pos = pos / max_signal_length
            norm_width = width / max_signal_length
            
            confidence = 1.0
            cell_index, offset = divmod(norm_pos * n_cells, 1)
            cell_index = int(cell_index)
            scaling_factor = norm_width / anchors[0]
            
            y[cell_index, 0] = confidence
            y[cell_index, 1] = offset
            y[cell_index, 2] = scaling_factor
        
        return torch.from_numpy(x), torch.from_numpy(y)
    


In [ ]:

# PyTorch Model
class PeakDetectionModel(nn.Module):
    def __init__(self):
        super(PeakDetectionModel, self).__init__()
        
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(2)
        
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(2)
        
        self.conv3 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool1d(2)
        
        self.conv4 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.relu4 = nn.ReLU()
        self.pool4 = nn.MaxPool1d(2)
        
        self.conv5 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.relu5 = nn.ReLU()
        self.pool5 = nn.MaxPool1d(2)
        
        self.conv6 = nn.Conv1d(128, n_anchors * n_outputs, kernel_size=3, padding=1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # Input shape: (batch, 1024, 1) -> transpose to (batch, 1, 1024)
        x = x.transpose(1, 2)
        
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        x = self.pool4(self.relu4(self.conv4(x)))
        x = self.pool5(self.relu5(self.conv5(x)))
        x = self.sigmoid(self.conv6(x))
        
        # Output shape: (batch, 3, 32) -> transpose to (batch, 32, 3)
        x = x.transpose(1, 2)
        
        return x


In [ ]:

# Loss function
class DetectionLoss(nn.Module):
    def __init__(self):
        super(DetectionLoss, self).__init__()
        self.neg_scale = 0.3
        self.pos_scale = 1 - self.neg_scale
    
    def forward(self, pred, true):
        neg_mask = 1 - true[..., 0]
        pos_mask = true[..., 0]
        
        error = (true - pred) ** 2
        confidence_error = error[..., 0]
        offset_error = error[..., 1]
        scaling_factor_error = error[..., 2]
        
        # Calculate loss for negative examples
        neg_confidence_loss = torch.sum(neg_mask * confidence_error)
        neg_loss = self.neg_scale * neg_confidence_loss
        
        # Calculate loss for positive examples
        pos_confidence_loss = torch.sum(pos_mask * confidence_error)
        pos_localization_loss = torch.sum(pos_mask * (offset_error + scaling_factor_error))
        pos_loss = self.pos_scale * (pos_confidence_loss + pos_localization_loss)
        
        batch_size = true.shape[0]
        return (neg_loss + pos_loss) / batch_size
    

In [ ]:

# Metric
class MeanAveragePrecisionMetric:
    def __init__(self, iou_thresh=0.5):
        self.iou_thresh = iou_thresh
        self.reset()
    
    def reset(self):
        self.tp = 0.0
        self.fp = 0.0
    
    def update(self, pred, true):
        true_confidence = true[..., 0]
        true_offset = true[..., 1]
        true_scaling_factor = true[..., 2]
        
        pred_confidence = pred[..., 0]
        pred_offset = pred[..., 1]
        pred_scaling_factor = pred[..., 2]
        
        # True and predicted class
        true_c = true_confidence
        pred_c = (pred_confidence >= 0.5).float()
        
        # True and predicted normalized position
        batch_size = true.shape[0]
        
        cell_index = torch.arange(0, n_cells, dtype=torch.float32, device=pred.device)
        cell_index = cell_index.unsqueeze(0).repeat(batch_size, 1)
        
        true_x = (cell_index + true_offset) / n_cells * max_signal_length
        pred_x = (cell_index + pred_offset) / n_cells * max_signal_length
        
        # True and predicted normalized width
        true_w = true_scaling_factor * anchors[0] * max_signal_length
        pred_w = pred_scaling_factor * anchors[0] * max_signal_length
        
        true_xmin = true_x - true_w / 2
        true_xmax = true_xmin + true_w
        true_size = true_xmax - true_xmin
        
        pred_xmin = pred_x - pred_w / 2
        pred_xmax = pred_xmin + pred_w
        pred_size = pred_xmax - pred_xmin
        
        inter_xmin = torch.maximum(true_xmin, pred_xmin)
        inter_xmax = torch.minimum(true_xmax, pred_xmax)
        inter_size = inter_xmax - inter_xmin
        
        union_size = true_size + pred_size - inter_size
        iou = inter_size / union_size
        
        tp = true_c * pred_c * (iou >= self.iou_thresh).float()
        tp = torch.sum(tp).item()
        
        fp = true_c * pred_c * (iou < self.iou_thresh).float() + (1.0 - true_c) * pred_c
        fp = torch.sum(fp).item()
        
        self.tp += tp
        self.fp += fp
    
    def compute(self):
        return self.tp / max(self.tp + self.fp, 1.0)
    

In [ ]:

# Training function
def train_epoch(model, dataloader, criterion, optimizer, metric, device):
    model.train()
    metric.reset()
    total_loss = 0.0
    
    for batch_x, batch_y in dataloader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        metric.update(outputs.detach(), batch_y)
    
    avg_loss = total_loss / len(dataloader)
    map_score = metric.compute()
    
    return avg_loss, map_score


# Validation function
def validate(model, dataloader, criterion, metric, device):
    model.eval()
    metric.reset()
    total_loss = 0.0
    
    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            
            total_loss += loss.item()
            metric.update(outputs, batch_y)
    
    avg_loss = total_loss / len(dataloader)
    map_score = metric.compute()
    
    return avg_loss, map_score


In [ ]:

# Main training script
def main():
    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Load data
    print("Loading raw data from disk...")
    info_data, signal_data = load_raw_covision_data("../data/info.raw", "../data/signal.raw")
    
    print("Extracting examples from raw data...")
    examples = raw_covision_data_to_examples(info_data, signal_data)
    print(f"Extracted {len(examples)} examples.")
    
    # Split data
    train_split = 0.6
    val_split = 0.2
    
    rng = np.random.default_rng(seed=0)
    rng.shuffle(examples)
    
    n_examples = len(examples)
    train_stop = int(n_examples * train_split)
    val_stop = train_stop + int(n_examples * val_split)
    
    train_examples = examples[:train_stop]
    val_examples = examples[train_stop:val_stop]
    test_examples = examples[val_stop:]
    
    # Create datasets and dataloaders
    train_dataset = CovisionDataset(train_examples)
    val_dataset = CovisionDataset(val_examples)
    test_dataset = CovisionDataset(test_examples)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
    
    # Initialize model, loss, optimizer, and metric
    model = PeakDetectionModel().to(device)
    criterion = DetectionLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)
    metric = MeanAveragePrecisionMetric()
    
    # Print model summary
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\nModel parameters: {total_params:,}")
    
    # Training loop
    num_epochs = 20
    print("\nStarting training...")
    
    for epoch in range(num_epochs):
        train_loss, train_map = train_epoch(model, train_loader, criterion, optimizer, metric, device)
        val_loss, val_map = validate(model, val_loader, criterion, metric, device)
        
        print(f"Epoch {epoch+1}/{num_epochs} - "
              f"loss: {train_loss:.4f} - map: {train_map:.4f} - "
              f"val_loss: {val_loss:.4f} - val_map: {val_map:.4f}")
    
    # Evaluate on test set
    print("\nEvaluating on test set...")
    test_loss, test_map = validate(model, test_loader, criterion, metric, device)
    print(f"Test - loss: {test_loss:.4f} - map: {test_map:.4f}")
    
    # Test on a single example
    model.eval()
    test_example = test_examples[123]
    
    batch_x = np.zeros((1, max_signal_length, 1), dtype=np.float32)
    batch_x[0, :len(test_example["signal"]), 0] = test_example["signal"]
    batch_x = torch.from_numpy(batch_x).to(device)
    
    with torch.no_grad():
        batch_p = model(batch_x).cpu().numpy()
    
    # Calculate predicted peaks
    peaks = []
    for cell_index, (confidence, offset, scaling_factor) in enumerate(batch_p[0]):
        if confidence > 0.5:
            position = (cell_index + offset) / n_cells * max_signal_length
            width = scaling_factor * anchors[0] * max_signal_length
            peaks.append([position, max_signal_height, width])
    
    print(f"\nGround truth peaks: {len(test_example['peaks'])}")
    print(f"Predicted peaks: {len(peaks)}")
    
    # Display results
    print("\nDisplaying ground truth...")
    display_example(test_example)
    
    print("Displaying prediction...")
    display_example({"signal": test_example["signal"], "peaks": np.array(peaks, np.float32)})


if __name__ == "__main__":
    main()
    

Using device: cuda
Loading raw data from disk...
Extracting examples from raw data...
Extracted 50000 examples.

Model parameters: 93,827

Starting training...
Epoch 1/20 - loss: 0.1987 - map: 0.4131 - val_loss: 0.0510 - val_map: 0.5035
Epoch 2/20 - loss: 0.0440 - map: 0.5394 - val_loss: 0.0354 - val_map: 0.6205
Epoch 3/20 - loss: 0.0341 - map: 0.6149 - val_loss: 0.0299 - val_map: 0.6828
Epoch 4/20 - loss: 0.0292 - map: 0.6594 - val_loss: 0.0251 - val_map: 0.7253
Epoch 5/20 - loss: 0.0256 - map: 0.7022 - val_loss: 0.0253 - val_map: 0.7378
Epoch 6/20 - loss: 0.0241 - map: 0.7199 - val_loss: 0.0230 - val_map: 0.7504
Epoch 7/20 - loss: 0.0229 - map: 0.7266 - val_loss: 0.0221 - val_map: 0.7947
Epoch 8/20 - loss: 0.0219 - map: 0.7358 - val_loss: 0.0204 - val_map: 0.8064
Epoch 9/20 - loss: 0.0208 - map: 0.7577 - val_loss: 0.0208 - val_map: 0.7854
Epoch 10/20 - loss: 0.0200 - map: 0.7651 - val_loss: 0.0192 - val_map: 0.8071
Epoch 11/20 - loss: 0.0200 - map: 0.7633 - val_loss: 0.0187 - val_map

Displaying prediction...
